## 1. Carga de Datos y Configuración inicial

In [ ]:
# Importación de Librerías

import sys
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Configuración del estilo visual general para los gráficos del EDA
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.titleweight"] = "bold"

In [ ]:
# Carga del dataset procesado
df = pd.read_csv('../data/processed/bank_marketing_cleaned.csv')

## 2. Análisis Univariado (Comportamiento Individual)

Se excluyen las variables:
- id_: Identificadores únicos
- date: Están sus componentes extraídos (contact_month, contact_year)
- Dt_Customer: Están sus componentes extraídos (Customer_year, Customer_month, Customer_tenure_year)

2.1. Variable Target (y): 
Balance de clases y porcentaje de respuesta a la campaña ("Suscribió" o "No Suscribió" un producto o servicio).

In [ ]:
# ------------------------------------------------------------------
# 2.1. ANÁLISIS UNIVARIADO: VARIABLE TARGET (y)
# ------------------------------------------------------------------

# Tabulación de frecuencias absolutas y relativas
target_counts = df['y'].value_counts()
target_percents = df['y'].value_counts(normalize=True) * 100

summary_target = pd.DataFrame({
    'Cantidad': target_counts,
    'Porcentaje (%)': target_percents.round(2)
})

print("=== DISTRIBUCIÓN DE LA VARIABLE TARGET (y) ===")
print(summary_target)

# Gráfico de barras con porcentajes
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='y', ax=ax, hue='y', palette='Set2', width=0.4, legend=False)

# Etiquetas con el porcentaje exacto sobre cada barra
total = len(df)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.2f}%'
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    ax.annotate(percentage, (x, y), ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.title('Distribución de la Variable Target (y)')
plt.xlabel('Respuesta a la Campaña (y)')
plt.ylabel('Cantidad de Clientes')
plt.ylim(0, total * 1.1)
plt.tight_layout()
plt.show()

2.2. Variables Numéricas: 
Histogramas y boxplots para evaluar sesgos, dispersión y outliers (ejemplo: Customer_tenure_year, Income, etc.).

In [ ]:
# ------------------------------------------------------------------
# 2.2. ANÁLISIS UNIVARIADO: VARIABLES NUMÉRICAS
# ------------------------------------------------------------------

# Identificar columnas numéricas
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Resumen estadístico descriptivo
print("=== RESUMEN ESTADÍSTICO DE VARIABLES NUMÉRICAS ===")
print(df[num_cols].describe().T[['mean', 'std', 'min', '50%', 'max']])

# Generación de Histograma y Boxplot lado a lado para cada variable
for col in num_cols:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    
    # Histograma para evaluar distribución y sesgo
    sns.histplot(data=df, x=col, kde=True, ax=axes[0], color='skyblue')
    axes[0].set_title(f'Distribución de {col}')
    axes[0].set_xlabel(col)
    axes[0].set_ylabel('Frecuencia')
    
    # Boxplot para detectar dispersión y outliers
    sns.boxplot(data=df, x=col, ax=axes[1], color='coral')
    axes[1].set_title(f'Dispersión y Outliers de {col}')
    axes[1].set_xlabel(col)
    
    plt.tight_layout()
    plt.show()

2.3. Variables Categóricas: 
Frecuencias relativas y distribución de las variables cualitativas.

In [ ]:
# ------------------------------------------------------------------
# 2.3. ANÁLISIS UNIVARIADO: VARIABLES CATEGÓRICAS
# ------------------------------------------------------------------

# Identificar columnas categóricas excluyendo 'y', 'id_', 'date', 'Dt_Customer'
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
cols_to_exclude = ['y', 'id_', 'date', 'Dt_Customer']
cat_cols = [col for col in cat_cols if col not in cols_to_exclude]

print(f"Procesando {len(cat_cols)} columnas categóricas: {cat_cols}\n")

# Resumen numérico y gráfico
for col in cat_cols:
    counts = df[col].value_counts()
    percents = df[col].value_counts(normalize=True) * 100
    
    summary = pd.DataFrame({
        'Cantidad': counts,
        'Porcentaje (%)': percents.round(2)
    })
    
    print(f"=== DISTRIBUCIÓN DE {col.upper()} ===")
    print(summary)
    print("\n")
    
    fig, ax = plt.subplots(figsize=(7, 3.5))
    sns.countplot(
        data=df, 
        y=col, 
        ax=ax, 
        hue=col, 
        palette='Set2', 
        order=counts.index, 
        legend=False
    )
    
    plt.title(f'Distribución de {col}')
    plt.xlabel('Cantidad de Clientes')
    plt.ylabel(col)
    plt.tight_layout()
    
    plt.show()
    plt.close(fig)

## 3. Análisis Bivariado (Relación con el Target "y")

Se excluyen las variables:
- id_: Identificadores únicos
- date: Están sus componentes extraídos (contact_month, contact_year)
- Dt_Customer: Están sus componentes extraídos (Customer_year, Customer_month, Customer_tenure_year)

### 3.1. Relación Numérica vs Target (y)

- 1: OBTENEMOS EL PORCENTAJE DE RESPUESTA DEL CLIENTE PARA CADA VARIABLE NUMÉRICA

In [ ]:
# ------------------------------------------------------------------
# 3. ANÁLISIS BIVARIADO: VARIABLES NUMÉRICAS VS TARGET (y)
# ------------------------------------------------------------------

# Identificar columnas numéricas a analizar
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols = [col for col in num_cols if col not in cols_to_exclude]

# Comparación de promedios según la respuesta del cliente
print("=== PROMEDIOS SEGÚN RESPUESTA A LA CAMPAÑA (y) ===")
print(df.groupby('y')[num_cols].mean().round(2).T)
print("\n")


- 2: DE ACUERDO A RESULTADO DE PORCENTAJES REALIZAREMOS UN ANÁLISIS PREDICTIVO VISUAL ADECUADO POR GRUPOS (detectamos que existen variables que no pueden evaluarse con un promedio simple)

- Grupo 1: Evaluación con Promedios Simples (Boxplots)

In [ ]:
# ----------------------------------------------------------------------------
# GRUPO 1: VARIABLES 'age', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx',
# 'euribor3m', # 'nr.employed' (BOXPLOTS)
# ----------------------------------------------------------------------------

# 1. Boxplots para variables macroeconómicas y edad
cols_boxplot = ['age', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(8, 8))
axes = axes.flatten()

for i, col in enumerate(cols_boxplot):
    if col in df.columns:
        sns.boxplot(
            data=df,
            x='y',
            y=col,
            hue='y',
            palette='Set2',
            ax=axes[i],
            legend=False
        )
        axes[i].set_title(f'Distribución de {col} según Target (y)')
        axes[i].set_xlabel('Respuesta a la Campaña (y)')
        axes[i].set_ylabel(col)

plt.tight_layout()
plt.show()
plt.close(fig)

# 2. Histograma doble con KDE para Income
if 'Income' in df.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(
        data=df,
        x='Income',
        hue='y',
        kde=True,
        element='step',
        palette='Set2',
        ax=ax
    )
    plt.title('Distribución e Impacto de Income según Target (y)')
    plt.xlabel('Income')
    plt.ylabel('Frecuencia')
    plt.tight_layout()
    plt.show()
    plt.close(fig)

- Grupo 2: Evaluación Independiente (Barras Agrupadas)

In [ ]:
# ------------------------------------------------------------------
# GRUPO 2: VARIABLE 'duration' (RANGOS Y BARRAS AGRUPADAS)
# ------------------------------------------------------------------

# 1. Agrupar 'duration' en rangos (valores expresados en segundos)
bins_duration = [0, 120, 300, 600, 1200, float('inf')]
labels_duration = ['0-2 min', '2-5 min', '5-10 min', '10-20 min', '+20 min']

df['duration_range'] = pd.cut(
    df['duration'], 
    bins=bins_duration, 
    labels=labels_duration, 
    include_lowest=True
)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.countplot(
    data=df,
    x='duration_range',
    hue='y',
    palette='Set2',
    ax=ax
)

plt.title('Respuesta a la Campaña (y) según Rango de Duración')
plt.xlabel('Rango de duration')
plt.ylabel('Cantidad de Clientes')
plt.legend(title='Suscripción (y)')
plt.tight_layout()

plt.show()
plt.close(fig)

In [ ]:
# ------------------------------------------------------------------
# GRUPO 2: VARIABLE 'campaign' (AGRUPACIÓN Y BARRAS AGRUPADAS)
# ------------------------------------------------------------------

# 1. Agrupar 'campaign'en rangos de llamadas
bins_campaign = [0, 2, 4, 6, 10, float('inf')]
labels_campaign = ['1-2 llamadas', '3-4 llamadas', '5-6 llamadas', '7-10 llamadas', '+10 llamadas']

df['campaign_grouped'] = pd.cut(
    df['campaign'],
    bins=bins_campaign,
    labels=labels_campaign,
    include_lowest=True
)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.countplot(
    data=df,
    x='campaign_grouped',
    hue='y',
    palette='Set2',
    ax=ax
)

plt.title('Respuesta a la Campaña (y) según Cantidad de Contactos')
plt.xlabel('Rango de campaign')
plt.ylabel('Cantidad de Clientes')
plt.legend(title='Suscripción (y)')
plt.tight_layout()

plt.show()
plt.close(fig)

In [ ]:
# -----------------------------------------------------------------------
# GRUPO 2: VARIABLES 'Kidhome', 'Teenhome' (CATEGÓRICAS Y BARRAS AGRUPADAS)
# -----------------------------------------------------------------------

# Función de categorización: 0 hijos, 1 hijo, 2+ hijos
def categorizar_hijos(val):
    if val == 0:
        return '0 hijos'
    elif val == 1:
        return '1 hijo'
    else:
        return '2+ hijos'

orden_hijos = ['0 hijos', '1 hijo', '2+ hijos']

for col in ['Kidhome', 'Teenhome']:
    if col in df.columns:
        # 1. Crear columna categórica
        cat_col = f'{col}_cat'
        df[cat_col] = df[col].apply(categorizar_hijos)
        df[cat_col] = pd.Categorical(df[cat_col], categories=orden_hijos, ordered=True)
        
        # 2. Tabla de frecuencias cruzadas (% de conversión por fila)
        cross_tab = pd.crosstab(df[cat_col], df['y'], normalize='index') * 100
        print(f"=== TABLA DE TARGET (y) SEGÚN {col.upper()} (%) ===")
        print(cross_tab.round(2))
        print("\n")
        
        # 3. Gráfico de Barras Agrupadas
        fig, ax = plt.subplots(figsize=(7, 4))
        sns.countplot(
            data=df,
            x=cat_col,
            hue='y',
            palette='Set2',
            ax=ax
        )
        
        plt.title(f'Respuesta a la Campaña (y) según {col}')
        plt.xlabel(f'Número de Hijos en el Hogar ({col})')
        plt.ylabel('Cantidad de Clientes')
        plt.legend(title='Suscripción (y)')
        plt.tight_layout()
        plt.show()
        plt.close(fig)

In [ ]:
# ------------------------------------------------------------------
# GRUPO 2: VARIABLE 'NumWebVisitsMonth' (AGRUPACIÓN Y BARRAS AGRUPADAS)
# ------------------------------------------------------------------

# 1. Agrupar 'NumWebVisitsMonth' en rangos de actividad digital
conditions = [
    (df['NumWebVisitsMonth'] <= 5),
    (df['NumWebVisitsMonth'] >= 6) & (df['NumWebVisitsMonth'] <= 15),
    (df['NumWebVisitsMonth'] > 15)
]

choices = ['Baja (0-5 visitas)', 'Media (6-15 visitas)', 'Alta (>15 visitas)']

df['web_visits_range'] = np.select(conditions, choices, default='Baja (0-5 visitas)')

# Ordenar lógicamente las categorías
order_visits = ['Baja (0-5 visitas)', 'Media (6-15 visitas)', 'Alta (>15 visitas)']
df['web_visits_range'] = pd.Categorical(df['web_visits_range'], categories=order_visits, ordered=True)

# 2. Gráfico de Barras Agrupadas
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.countplot(
    data=df,
    x='web_visits_range',
    hue='y',
    palette='Set2',
    ax=ax
)

plt.title('Respuesta a la Campaña (y) según Actividad Web Mensual')
plt.xlabel('Nivel de NumWebVisitsMonth')
plt.ylabel('Cantidad de Clientes')
plt.legend(title='Suscripción (y)')
plt.tight_layout()

plt.show()
plt.close(fig)

- Grupo 3: Evaluación Independiente (Barras Apiladas al 100%)

In [ ]:
# ------------------------------------------------------------------
# GRUPO 3: VARIABLE 'pdays'(AGRUPACIÓN Y BARRAS APILADAS AL 100%)
# ------------------------------------------------------------------

# 1. Categorizar pdays
# (Contempla -1 y 999 como 'Nunca contactado', según la versión del dataset)
conditions = [
    (df['pdays'] == -1) | (df['pdays'] == 999),
    (df['pdays'] == 0),
    (df['pdays'] >= 1) & (df['pdays'] <= 3),
    (df['pdays'] >= 4) & (df['pdays'] <= 7),
    (df['pdays'] > 7) & (df['pdays'] != 999)
]

choices = ['Nunca contactado', '0 días', '1-3 días', '4-7 días', '>7 días']

df['pdays_range'] = np.select(conditions, choices, default='Nunca contactado')

# Definir el orden de las categorías para el gráfico
order_pdays = ['Nunca contactado', '0 días', '1-3 días', '4-7 días', '>7 días']
df['pdays_range'] = pd.Categorical(df['pdays_range'], categories=order_pdays, ordered=True)

# 2. Calcular porcentajes proporcionales (Normalizado por fila al 100%)
cross_pdays = pd.crosstab(df['pdays_range'], df['y'], normalize='index') * 100

# 3. Gráfico de Barras Apiladas al 100%
ax = cross_pdays.plot(
    kind='bar', 
    stacked=True, 
    figsize=(8, 4.5), 
    colormap='Set2'
)

plt.title('Respuesta a la Campaña (y) según Días desde Último Contacto')
plt.xlabel('Rango de pdays')
plt.ylabel('Porcentaje (%)')
plt.legend(title='Suscripción (y)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)

# Agregar etiquetas numéricas dentro de cada segmento de la barra
for p in ax.patches:
    height = p.get_height()
    if height > 3:  # Solo muestra la etiqueta si la barra es lo suficientemente ancha
        ax.annotate(
            f'{height:.1f}%',
            (p.get_x() + p.get_width() / 2., p.get_y() + height / 2.),
            ha='center', 
            va='center',
            fontsize=9,
            color='white',
            fontweight='bold'
        )

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# ------------------------------------------------------------------
# GRUPO 3: VARIABLE 'previous' (AGRUPACIÓN Y BARRAS APILADAS AL 100%)
# ------------------------------------------------------------------

# 1. Categorizar 'previous'
conditions = [
    df['previous'] == 0,
    df['previous'] == 1,
    df['previous'] == 2,
    df['previous'] > 2
]

choices = ['Ningún contacto', '1 contacto', '2 contactos', '> 2 contactos']

df['previous_range'] = np.select(conditions, choices, default='Ningún contacto previo')

# Ordenar lógicamente las categorías
order_prev = ['Ningún contacto', '1 contacto', '2 contactos', '> 2 contactos']
df['previous_range'] = pd.Categorical(df['previous_range'], categories=order_prev, ordered=True)

# 2. Calcular tabla cruzada en porcentajes por fila (100%)
cross_prev = pd.crosstab(df['previous_range'], df['y'], normalize='index') * 100

# 3. Gráfico de Barras Apiladas al 100%
ax = cross_prev.plot(
    kind='bar', 
    stacked=True, 
    figsize=(8, 4.5), 
    colormap='Set2'
)

plt.title('Respuesta a la Campaña (y) según Contactos Previos')
plt.xlabel('Historial de previous')
plt.ylabel('Porcentaje (%)')
plt.legend(title='Suscripción (y)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)

# Etiquetas numéricas dentro de los segmentos de barra
for p in ax.patches:
    height = p.get_height()
    if height > 3:
        ax.annotate(
            f'{height:.1f}%',
            (p.get_x() + p.get_width() / 2., p.get_y() + height / 2.),
            ha='center', 
            va='center',
            fontsize=9,
            color='white',
            fontweight='bold'
        )

plt.tight_layout()
plt.show()
plt.close()

- Grupo 4: Evaluación Variables Geográficas (Dispersión Geográfica)

In [ ]:
# ------------------------------------------------------------------
# GRUPO 4: VARIABLES 'latitude', 'longitude'(DISPERSIÓN GEOGRÁFICA)
# ------------------------------------------------------------------

# Identificar nombres de columnas de latitud y longitud en el dataset
lat_col = [col for col in df.columns if col.lower() in ['latitude', 'lat']][0]
lon_col = [col for col in df.columns if col.lower() in ['longitude', 'lon', 'lng']][0]

# Tomar una muestra aleatoria de 5.000 filas para evitar la saturación (overplotting)
df_sample = df.sample(n=min(5000, len(df)), random_state=42)

# Gráfico de Dispersión Geográfico
fig, ax = plt.subplots(figsize=(10, 6))

sns.scatterplot(
    data=df_sample,
    x=lon_col,
    y=lat_col,
    hue='y',
    palette={'no': '#e74c3c', 'yes': '#2ecc71'},
    alpha=0.5,
    s=25,
    ax=ax
)

plt.title(f'Distribución Geográfica de Clientes (Muestra Aleatoria de {len(df_sample):,} filas)')
plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.legend(title='Suscripción (y)', loc='best')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()
plt.close(fig)

- Grupo 5: Evaluación Variables Temporales (Lineal)

In [ ]:
# ------------------------------------------------------------------------------
# 5.1 VARIABLES TEMPORALES: contact_month Y contact_year (LÍNEAS)
# ------------------------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. ESTACIONALIDAD MENSUAL (contact_month)
if 'contact_month' in df.columns:
    # Tabla de contingencia porcentual (% por mes)
    tabla_mes = pd.crosstab(df['contact_month'], df['y'], normalize='index') * 100
    tabla_mes = tabla_mes.sort_index()

    print("=== TABLA DE TARGET (y) SEGUN MES (%) ===")
    print(tabla_mes.round(2))
    print("\n")

    if 'yes' in tabla_mes.columns:
        x_months = [str(m) for m in tabla_mes.index]
        y_months = tabla_mes['yes'].values

        axes[0].plot(x_months, y_months, marker='o', color='#27ae60', linewidth=2.5, markersize=8)
        
        # Anotar porcentajes sobre los puntos
        for i, val in enumerate(y_months):
            axes[0].annotate(f'{val:.1f}%', (i, val), xytext=(0, 8), 
                             textcoords='offset points', ha='center', fontweight='bold', fontsize=9)

    axes[0].set_title('Respuesta a la Campaña según Mes de Contacto', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('contact_month')
    axes[0].set_ylabel('Suscripción (y)')
    axes[0].legend(title='Suscripción (%yes)')
    axes[0].grid(True, linestyle='--', alpha=0.5)

# 2. TENDENCIA ANUAL (contact_year)
if 'contact_year' in df.columns:
    # Tabla de contingencia porcentual (% por año)
    tabla_anio = pd.crosstab(df['contact_year'], df['y'], normalize='index') * 100
    tabla_anio = tabla_anio.sort_index()

    print("=== TABLA DE TARGET (y) SEGUN AÑO (%) ===")
    print(tabla_anio.round(2))

    if 'yes' in tabla_anio.columns:
        x_years = [str(y) for y in tabla_anio.index]
        y_years = tabla_anio['yes'].values

        axes[1].plot(x_years, y_years, marker='s', color='#2980b9', linewidth=2.5, markersize=8)
        
        # Anotar porcentajes sobre los puntos
        for i, val in enumerate(y_years):
            axes[1].annotate(f'{val:.1f}%', (i, val), xytext=(0, 8), 
                             textcoords='offset points', ha='center', fontweight='bold', fontsize=9)

    axes[1].set_title('Respuesta a la Campaña según Año de Contacto', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('contact_year')
    axes[1].set_ylabel('Suscripción (y)')
    axes[1].legend(title='Suscripción (%yes)')
    axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
# ------------------------------------------------------------------------------
# 5.2 VARIABLES TEMPORALES: Customer_month, Customer_year,Customer_tenure_year
# ------------------------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. MES DE ALTA DEL CLIENTE (Customer_month)
if 'Customer_month' in df.columns:
    tabla_cust_mes = pd.crosstab(df['Customer_month'], df['y'], normalize='index') * 100
    tabla_cust_mes = tabla_cust_mes.sort_index()

    print("=== TABLA DE TARGET (y) SEGUN MES DE ALTA (%) ===")
    print(tabla_cust_mes.round(2))
    print("\n")

    if 'yes' in tabla_cust_mes.columns:
        x_vals = [str(m) for m in tabla_cust_mes.index]
        y_vals = tabla_cust_mes['yes'].values

        axes[0].plot(
            x_vals, 
            y_vals, 
            marker='o', 
            color='#27ae60', 
            linewidth=2.5, 
            markersize=8, 
            label='Suscripción (% yes)'
        )
        
        for i, val in enumerate(y_vals):
            axes[0].annotate(f'{val:.1f}%', (i, val), xytext=(0, 8), 
                             textcoords='offset points', ha='center', fontweight='bold', fontsize=9)

    axes[0].set_title('Respuesta a la Campaña según Mes de Alta del Cliente', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Customer_month')
    axes[0].set_ylabel('Suscripción (y)')
    axes[0].legend(title='Suscripción (%yes)')
    axes[0].grid(True, linestyle='--', alpha=0.5)

# 2. AÑO DE ALTA DEL CLIENTE (Customer_year)
if 'Customer_year' in df.columns:
    tabla_cust_anio = pd.crosstab(df['Customer_year'], df['y'], normalize='index') * 100
    tabla_cust_anio = tabla_cust_anio.sort_index()

    print("=== TABLA DE TARGET (y) SEGUN AÑO DE ALTA (%) ===")
    print(tabla_cust_anio.round(2))
    print("\n")

    if 'yes' in tabla_cust_anio.columns:
        x_vals = [str(y) for y in tabla_cust_anio.index]
        y_vals = tabla_cust_anio['yes'].values

        axes[1].plot(
            x_vals, 
            y_vals, 
            marker='s', 
            color='#2980b9', 
            linewidth=2.5, 
            markersize=8, 
            label='Suscripción (% yes)'
        )
        
        for i, val in enumerate(y_vals):
            axes[1].annotate(f'{val:.1f}%', (i, val), xytext=(0, 8), 
                             textcoords='offset points', ha='center', fontweight='bold', fontsize=9)

    axes[1].set_title('Respuesta a la Campaña según Año de Alta del Cliente', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Customer_year')
    axes[1].set_ylabel('Suscripción (y)')
    axes[1].legend(title='Suscripción (%yes)')
    axes[1].grid(True, linestyle='--', alpha=0.5)

# 3. ANTIGÜEDAD DEL CLIENTE (Customer_tenure_year)
if 'Customer_tenure_year' in df.columns:
    tabla_tenure = pd.crosstab(df['Customer_tenure_year'], df['y'], normalize='index') * 100
    tabla_tenure = tabla_tenure.sort_index()

    print("=== TABLA DE TARGET (y) SEGUN ANTIGÜEDAD (%) ===")
    print(tabla_tenure.round(2))
    print("\n") 

    if 'yes' in tabla_tenure.columns:
        x_vals = [str(t) for t in tabla_tenure.index]
        y_vals = tabla_tenure['yes'].values

        axes[2].plot(
            x_vals, 
            y_vals, 
            marker='^', 
            color='#8e44ad', 
            linewidth=2.5, 
            markersize=8, 
            label='Suscripción (% yes)'
        )
        
        for i, val in enumerate(y_vals):
            axes[2].annotate(f'{val:.1f}%', (i, val), xytext=(0, 8), 
                             textcoords='offset points', ha='center', fontweight='bold', fontsize=9)

    axes[2].set_title('Respuesta a la Campaña según Antigüedad del Cliente', fontsize=12, fontweight='bold')
    axes[2].set_xlabel('Customer_tenure_year')
    axes[2].set_ylabel('Suscripción (y)')
    axes[2].legend(title='Suscripción (%yes)')
    axes[2].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()
plt.close(fig)

### 3.2. Relación Categórica vs Target (y)

Se excluyen las variables:
- id_: Identificadores únicos
- date
- Dt_Customer

In [ ]:
# ------------------------------------------------------------------------------
# ANÁLISIS BIVARIADO: VARIABLES CATEGÓRICAS Y TASA DE CONVERSIÓN
# ------------------------------------------------------------------------------

# Lista de variables categóricas principales del dataset
cols_cat = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'poutcome']

# Filtrar solo las columnas que existen en el DataFrame
cols_analizar = [col for col in cols_cat if col in df.columns]

for col in cols_analizar:
    # 1. CÁLCULO DE TABLA DE CONTINGENCIA Y FRECUENCIAS
    conteo = df[col].value_counts()
    tabla_perc = pd.crosstab(df[col], df['y'], normalize='index') * 100
    
    # Unir volumen total y porcentaje de conversión
    resumen = pd.DataFrame({
        'Total_Clientes': conteo,
        'no%': tabla_perc['no'] if 'no' in tabla_perc.columns else 0,
        'yes%': tabla_perc['yes'] if 'yes' in tabla_perc.columns else 0
    })
    
    # Ordenar de mayor a menor tasa de conversión
    resumen = resumen.sort_values(by='yes%', ascending=False)

    print(f"==================================================")
    print(f"  TABLA DE TARGET (y) SEGÚN {col.upper()}")
    print(f"==================================================")
    print(resumen.round(2))
    print("\n")

    # 2. GRAFICA DE TASA DE CONVERSIÓN
    fig, ax = plt.subplots(figsize=(10, 4))
    
    # Gráfico de barras horizontales para facilitar lectura de etiquetas
    bars = ax.barh(resumen.index.astype(str), resumen['yes%'], color="#db8d34", edgecolor='black', alpha=0.8)
    
    # Invertir eje Y para mostrar la mayor conversión arriba
    ax.invert_yaxis()
    
    # Anotar porcentaje al final de cada barra
    for bar in bars:
        width = bar.get_width()
        ax.annotate(f'{width:.1f}%',
                    xy=(width, bar.get_y() + bar.get_height() / 2),
                    xytext=(5, 0),
                    textcoords="offset points",
                    ha='left', va='center', fontweight='bold', fontsize=9)

    ax.set_title(f'Target (y) según {col}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Suscripción (% yes)')
    ax.set_ylabel(col)
    ax.grid(True, linestyle='--', alpha=0.5, axis='x')
    
    plt.tight_layout()
    plt.show()
    plt.close(fig)

## 4. Análisis Multivariado y Correlaciones

### 4.1. Matriz de correlación (Heatmap) entre variables numéricas para identificar colinealidad.	

In [ ]:
# ------------------------------------------------------------------------------
# ANÁLISIS MULTIVARIADO: MATRIZ DE CORRELACIÓN (HEATMAP)
# ------------------------------------------------------------------------------

# 1. Seleccionar variables numéricas
df_num = df.select_dtypes(include=['number']).copy()

# Si 'y' no es numérica, agregamos una columna binaria para evaluar su correlación
if 'y' in df.columns and 'target_y' not in df_num.columns:
    df_num['target_y'] = (df['y'] == 'yes').astype(int)

# 2. Calcular la matriz de correlación de Pearson
corr_matrix = df_num.corr()

# 3. Crear máscara para la mitad superior (evita redundancia)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# 4. Graficar Heatmap
plt.figure(figsize=(11, 9))

sns.heatmap(
    corr_matrix, 
    mask=mask, 
    annot=True, 
    fmt='.2f', 
    cmap='coolwarm', 
    vmin=-1, 
    vmax=1, 
    center=0,
    square=True, 
    linewidths=0.8, 
    cbar_kws={"shrink": 0.8}
)

plt.title('Matriz de Correlación de Variables Numéricas', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# 5. Identificar pares con alta colinealidad (|r| > 0.7)
print("=== POSIBLES PROBLEMAS DE COLINEALIDAD (|r| > 0.7) ===")
alta_colinealidad = False

for i in range(len(corr_matrix.columns)):
    for j in range(i):
        coef = corr_matrix.iloc[i, j]
        if abs(coef) >= 0.7:
            var1 = corr_matrix.columns[i]
            var2 = corr_matrix.columns[j]
            print(f"- {var1} <---> {var2}: r = {coef:.2f}")
            alta_colinealidad = True

if not alta_colinealidad:
    print("No se detectaron pares de variables numéricas con alta colinealidad (r >= 0.7).")

 El resultado de Colinealidad indica que existen variables numéricas altamente redundantes (multicolinealidad, con r > 0.7). Significa que varias columnas están aportando exactamente el mismo patrón de información al dataset.

### 4.2. Interacción Multivariada de Factores Clave	

- Definición de Factores Claves para el análisis de interacción

In [ ]:
# Definición de factores clave para el análisis de interacción
factores_clave = {
    'demograficos': ['job', 'education', 'age_group'],
    'comportamiento': ['poutcome', 'contact', 'previous'],
    'macroeconomico': ['euribor3m'],  # Representante elegido del bloque económico
    'temporal': ['Customer_year', 'contact_month']
}

1. INTERACCIÓN:  **job**  con  **poutcome**

In [ ]:
# 1. INTERACCIÓN: Ocupación (job) x Resultado de Campaña Previa (poutcome)
plt.figure(figsize=(10, 6))
inter_job_poutcome = df.pivot_table(
    index='job', 
    columns='poutcome', 
    values='y', 
    aggfunc=lambda x: (x == 'yes').mean() * 100
)

sns.heatmap(inter_job_poutcome, annot=True, fmt='.1f', cmap='YlGnBu', cbar_kws={'label': 'Suscripción (% yes)'})
plt.title('Interacción: Ocupación vs Resultado de Campaña Previa', fontsize=12, fontweight='bold')
plt.xlabel('Resultado de Campaña Previa (poutcome)')
plt.ylabel('Ocupación (job)')
plt.tight_layout()
plt.show()

2. INTERACCIÓN:  **euribor3m**  con  **contact**

In [ ]:
# 2. INTERACCIÓN: Tasa de Interés Trimestral (euribor3m) x Medio de Contacto (contact)

# Segmentar euribor3m en 3 niveles (Bajo, Medio, Alto)
df['euribor_nivel'] = pd.qcut(df['euribor3m'], q=3, labels=['Bajo', 'Medio', 'Alto'])

# Crear la tabla pivote
inter_euribor_contact = df.pivot_table(
    index='euribor_nivel', 
    columns='contact', 
    values='y', 
    aggfunc=lambda x: (x == 'yes').mean() * 100
)

# Graficar Heatmap
plt.figure(figsize=(8, 5))
sns.heatmap(
    inter_euribor_contact, 
    annot=True, 
    fmt='.1f', 
    cmap='Blues', 
    cbar_kws={'label': 'Suscripción (% yes)'}
)

plt.title('Interacción: Tasa de Interés Trimestral vs Medio de Contacto', fontsize=12, fontweight='bold')
plt.xlabel('Medio de Contacto (contact)')
plt.ylabel('Nivel de Tasa de Interés Trimestral (euribor3m)')
plt.tight_layout()
plt.show()

3. INTERACCIÓN:  **job**  con  **education**

In [ ]:
# 3. INTERACCIÓN: Ocupación (job) x Nivel Educativo (education)
plt.figure(figsize=(12, 6))
inter_job_edu = df.pivot_table(
    index='job', 
    columns='education', 
    values='y', 
    aggfunc=lambda x: (x == 'yes').mean() * 100
)

sns.heatmap(inter_job_edu, annot=True, fmt='.1f', cmap='Greens', cbar_kws={'label': 'Suscripción (% yes)'})
plt.title('Interacción: Ocupación vs Nivel Educativo', fontsize=12, fontweight='bold')
plt.xlabel('Nivel Educativo (education)')
plt.ylabel('Ocupación (job)')
plt.tight_layout()
plt.show()

4. INTERACCIÓN:  **age**  con  **poutcome**

In [ ]:
# 4. INTERACCIÓN: Edad del Cliente (age) x Resultado de Campaña Previa (poutcome)

# Crear rango etario si no existe en el DataFrame
if 'age_group' not in df.columns:
    df['age_group'] = pd.cut(df['age'], bins=[0, 30, 45, 60, 100], labels=['<30', '30-45', '46-60', '>60'])

plt.figure(figsize=(9, 5))
inter_age_poutcome = df.pivot_table(
    index='age_group', 
    columns='poutcome', 
    values='y', 
    aggfunc=lambda x: (x == 'yes').mean() * 100
)

sns.heatmap(inter_age_poutcome, annot=True, fmt='.1f', cmap='Oranges', cbar_kws={'label': 'Suscripción (% yes)'})
plt.title('Interacción: Edad del Cliente vs Resultado de Campaña Previa', fontsize=12, fontweight='bold')
plt.xlabel('Resultado de Campaña Previa (poutcome)')
plt.ylabel('Edad del Cliente (age_group)')
plt.tight_layout()
plt.show()

5. INTERACCIÓN:  **loan**  con  **poutcome**

In [ ]:
# 7. INTERACCIÓN: Préstamo Personal (loan) x Resultado de Campaña Previa (poutcome)

# Evalúa si tener un préstamo frena la conversión incluso en clientes con historial positivo
plt.figure(figsize=(8, 5))
inter_loan_poutcome = df.pivot_table(
    index='loan', 
    columns='poutcome', 
    values='y', 
    aggfunc=lambda x: (x == 'yes').mean() * 100
)

sns.heatmap(inter_loan_poutcome, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': 'Suscripción (% yes)'})
plt.title('Interacción: Préstamo Personal vs Resultado de Campaña Previa', fontsize=12, fontweight='bold')
plt.xlabel('Resultado de Campaña Previa (poutcome)')
plt.ylabel('Préstamo Personal (loan)')
plt.tight_layout()
plt.show()

6. INTERACCIÓN:  **Customer_tenure_year**  con  **poutcome**

In [ ]:
# 8. INTERACCIÓN: Antigüedad del Cliente (Customer_tenure_year) x Resultado de Campaña Previa (poutcome)
plt.figure(figsize=(9, 5))
inter_tenure_poutcome = df.pivot_table(
    index='Customer_tenure_year', 
    columns='poutcome', 
    values='y', 
    aggfunc=lambda x: (x == 'yes').mean() * 100
)

sns.heatmap(inter_tenure_poutcome, annot=True, fmt='.1f', cmap='YlGn', cbar_kws={'label': 'Suscripción (% yes)'})
plt.title('Interacción: Antigüedad del Cliente vs Resultado de Campaña Previa', fontsize=12, fontweight='bold')
plt.xlabel('Resultado de Campaña Previa (poutcome)')
plt.ylabel('Antigüedad del Cliente (Customer_tenure_year)')
plt.tight_layout()
plt.show()

7. INTERACCIÓN:  **Customer_tenure_year**  con  **loan**

In [ ]:
# 9. INTERACCIÓN: Antigüedad del Cliente (Customer_tenure_year) x Préstamo Personal (loan)
plt.figure(figsize=(8, 5))
inter_tenure_loan = df.pivot_table(
    index='Customer_tenure_year', 
    columns='loan', 
    values='y', 
    aggfunc=lambda x: (x == 'yes').mean() * 100
)

sns.heatmap(inter_tenure_loan, annot=True, fmt='.1f', cmap='BuPu', cbar_kws={'label': 'Suscripción (% yes)'})
plt.title('Interacción: Antigüedad del Cliente vs Préstamo Personal', fontsize=12, fontweight='bold')
plt.xlabel('Préstamo Personal (loan)')
plt.ylabel('Antigüedad del Cliente (Customer_tenure_year)')
plt.tight_layout()
plt.show()

## 5: Síntesis de Insights e Hipótesis de Negocio

---

**1. Resumen de Hallazgos Clave y Patrones de Comportamiento**

* **Historial Previo (`poutcome`):** Constituye el factor predictivo con mayor impacto individual. Los clientes con conversiones previas exitosas presentan una tasa de respuesta significativamente más alta en comparación con nuevos prospectos.
* **Sensibilidad Macroeconómica (`euribor3m`):** La contratación de depósitos a plazo muestra alta sensibilidad al entorno financiero. Los periodos de tasas interbancarias bajas registran un aumento notable en la disposición a la conversión (`y = yes`).
* **Perfil Socioeconómico (`job`, `education`):** Los segmentos de jubilados y estudiantes lideran la receptividad comercial, superando a los perfiles administrativos y corporativos.
* **Carga Financiera (`loan`, `housing`):** La presencia de un préstamo personal no anula la conversión, pero la combinación de múltiples créditos activos reduce la respuesta positiva en segmentos con ingresos moderados.

---

**2. Selección de Variables con Mayor Potencial Predictivo**

**Variables Seleccionadas para el Modelo:**
* `poutcome`: Métrica clave de comportamiento histórico.
* `euribor3m`: Representante sintético del bloque macroeconómico.
* `job`: Variable categórica para segmentación socioeconómica.
* `age_group` / `age`: Captura el comportamiento según etapas de vida.
* `Customer_tenure_year`: Variable discreta de antigüedad y fidelidad del cliente.
* `loan` / `housing`: Indicadores de compromiso financiero actual.

**Variables Excluidas por Multicolinealidad ($|r| > 0.7$):**
* `emp.var.rate`, `nr.employed` y `cons.price.idx`: Excluidas por su elevada correlación con `euribor3m`, evitando redundancia y distorsión de coeficientes en algoritmos.
* `contact_year`: Excluida debido a su dependencia matemática directa con `Customer_tenure_year`.

---

**3. Hipótesis de Negocio y Recomendaciones**

* **$H_1$ (Priorización por Historial):** Focalizar la contactabilidad en clientes con `poutcome = success` maximizará el retorno comercial consumiendo menor volumen de llamadas.
* **$H_2$ (Oportunidad por Contexto):** Intensificar el esfuerzo comercial de captación durante ventanas temporales con tendencia a la baja en la tasa `euribor3m`.
* **$H_3$ (Estrategia Segmentada):** Desarrollar productos a medida para los segmentos de estudiantes y jubilados permitirá capturar volumen en los perfiles con mayor tasa de conversión natural.